# Exploratory Data Analysis (EDA) - Credit Risk Scorer

This notebook contains the exploratory analysis of the "Give Me Some Credit" dataset to profile features, check missing values, duplicates, outliers, and skewness.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import DataLoader
from src.config import RAW_DATA_PATH

sns.set_theme(style="whitegrid")

## 1. Load Data

In [ ]:
loader = DataLoader(RAW_DATA_PATH)
df = loader.load_data()
df.head()

## 2. General Dataset Diagnostics (Shapes, Missing Values, Duplicates)

In [ ]:
print(f"Shape: {df.shape}")
print(f"Total duplicates: {df.duplicated().sum()}")

missing_info = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Percentage': df.isnull().mean() * 100
})
missing_info

### Ingestion Diagnostics:
- **Duplicates**: 609 duplicate rows are present.
- **Missing Values**: `MonthlyIncome` is missing 19.82% of its entries. `NumberOfDependents` is missing 2.62% of entries. Other features are fully populated.

## 3. Target Class Imbalance

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x="SeriousDlqin2yrs", data=df, hue="SeriousDlqin2yrs", palette="muted", legend=False)
plt.title("Target Class Distribution")
plt.xlabel("Serious Delinquency (2 Years)")
plt.ylabel("Count")

total = len(df)
for p in ax.patches:
    pct = f"{100 * p.get_height() / total:.2f}%"
    ax.annotate(pct, (p.get_x() + p.get_width() / 2., p.get_height() + 1500), ha="center")
plt.show()

### Target Balance Insight:
- **Negative Class (0)**: 93.32% (did not experience financial distress)
- **Positive Class (1)**: 6.68% (experienced financial distress)
- **Action Item**: Accuracy is a misleading metric here. We will prioritize ROC-AUC, F1-score, and precision-recall metrics.

## 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", square=True, linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

### Correlation Insights:
- The three delinquency counter features (`NumberOfTime30-59DaysPastDueNotWorse`, `NumberOfTimes90DaysLate`, `NumberOfTime60-89DaysPastDueNotWorse`) show extremely high correlation (>0.98).
- **Reason**: This is due to a data anomaly (use of 96 or 98 as a status placeholder for late payments).

## 5. Feature Distributions & Skewness

In [ ]:
print("Feature Skewness:")
print(df.skew())

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sns.histplot(df["age"], bins=30, kde=True, ax=axes[0, 0], color="skyblue")
axes[0, 0].set_title("Age Distribution")

sns.histplot(df["NumberOfOpenCreditLinesAndLoans"], bins=20, kde=True, ax=axes[0, 1], color="salmon")
axes[0, 1].set_title("Open Credit Lines")

income_filt = df["MonthlyIncome"].dropna()
sns.histplot(income_filt[income_filt < 25000], bins=30, kde=True, ax=axes[1, 0], color="lightgreen")
axes[1, 0].set_title("Monthly Income (<25k)")

debt_filt = df[df["DebtRatio"] < 2]["DebtRatio"]
sns.histplot(debt_filt, bins=30, kde=True, ax=axes[1, 1], color="gold")
axes[1, 1].set_title("Debt Ratio (<2)")
plt.tight_layout()
plt.show()

### Skewness Insights:
- `age` is normally distributed (skewness ~0.19).
- `MonthlyIncome` and `DebtRatio` are extremely right-skewed (skews of >90). Log transformations in Phase 5 will be useful to stabilize the variance.

## 6. Outlier Visualization

In [ ]:
cols = ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome"]
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
for i, col in enumerate(cols):
    sns.boxplot(x=df[col].dropna(), ax=axes[i], color="lightcoral")
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

### Outlier Insights:
- `RevolvingUtilizationOfUnsecuredLines` has values up to 50,000+. Conceptually, utilization should be between 0 and 1. High values are either errors or extreme outliers.
- `DebtRatio` has values up to 329,663. For records missing `MonthlyIncome`, the raw debt amount was recorded directly in `DebtRatio`, which explains these values.
- We will address these issues in Phase 4: Data Cleaning.